In [35]:
import re
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("../Data/customer_reviews_clean.csv")
df.head()

,id,asins,product_name,title,text,rating,do_recommend,keys,clean_text,n_words,sentiment
0,AVqkIhwDv8e3D1O-lebb,B01AHB9CN2,all new fire hd 8 tablet 8 hd display wi fi 16...,Kindle,This product so far has not disappointed. My c...,5,1.0,"841667104676,amazon/53004484,amazon/b01ahb9cn2...",kindle . this product so far has not disappoin...,29,positive
1,AVqkIhwDv8e3D1O-lebb,B01AHB9CN2,all new fire hd 8 tablet 8 hd display wi fi 16...,very fast,great for beginner or experienced person. Boug...,5,1.0,"841667104676,amazon/53004484,amazon/b01ahb9cn2...",very fast . great for beginner or experienced ...,17,positive
2,AVqkIhwDv8e3D1O-lebb,B01AHB9CN2,all new fire hd 8 tablet 8 hd display wi fi 16...,Beginner tablet for our 9 year old son.,Inexpensive tablet for him to use and learn on...,5,1.0,"841667104676,amazon/53004484,amazon/b01ahb9cn2...",beginner tablet for our 9 year old son. . inex...,35,positive
3,AVqkIhwDv8e3D1O-lebb,B01AHB9CN2,all new fire hd 8 tablet 8 hd display wi fi 16...,Good!!!,I've had my Fire HD 8 two weeks now and I love...,4,1.0,"841667104676,amazon/53004484,amazon/b01ahb9cn2...",good!! . i've had my fire hd 8 two weeks now a...,120,positive
4,AVqkIhwDv8e3D1O-lebb,B01AHB9CN2,all new fire hd 8 tablet 8 hd display wi fi 16...,Fantastic Tablet for kids,I bought this for my grand daughter when she c...,5,1.0,"841667104676,amazon/53004484,amazon/b01ahb9cn2...",fantastic tablet for kids . i bought this for ...,122,positive


In [36]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("datafiniti/consumer-reviews-of-amazon-products")

print("Path to dataset files:", path)

import os
import pandas as pd

print(os.listdir(path))  # see which files are there
df_original = pd.read_csv(os.path.join(path, "1429_1.csv"), usecols=["id", "categories"], low_memory=False)

# One categories value per product
cat_by_id = df_original.drop_duplicates("id").set_index("id")["categories"]

# Add it back to your cleaned data
df["categories"] = df["id"].map(cat_by_id)

print(df["categories"].isna().sum())   
df.dtypes

Path to dataset files: C:\Users\PCACER\.cache\kagglehub\datasets\datafiniti\consumer-reviews-of-amazon-products\versions\5
['1429_1.csv', 'Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products.csv', 'Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products_May19.csv']
0


id                  str
asins               str
product_name        str
title               str
text                str
rating            int64
do_recommend    float64
keys                str
clean_text          str
n_words           int64
sentiment           str
categories          str
dtype: object

In [37]:
from sklearn.feature_extraction.text import TfidfVectorizer

products = df.drop_duplicates("product_name")[["product_name", "categories"]]
text = products["product_name"] + " " + products["categories"]   # name + categories

vectorizer = TfidfVectorizer(stop_words="english")
X = vectorizer.fit_transform(text)     # one row of numbers per product

print(X.shape)   # (number of products, number of words)

(38, 226)


In [38]:
products = df.drop_duplicates("product_name")[["product_name", "categories"]]
text = products["product_name"] + " " + products["categories"]

In [39]:
vectorizer = TfidfVectorizer(stop_words="english")
X = vectorizer.fit_transform(text)     # one row of numbers per product

print(X.shape)   # (number of products, number of words)
X

(38, 226)


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 809 stored elements and shape (38, 226)>

In [49]:
from sklearn.cluster import KMeans

from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=5, n_init=10, random_state=42)
products["cluster"] = kmeans.fit_predict(X)

for number, group in products.groupby("cluster"):
    print(f"\n--- cluster {number} ({len(group)} products)")
    for row in group.sort_values("n_reviews", ascending=False).itertuples():
        print(f"{row.n_reviews:>6}  {row.product_name[:70]}")


--- cluster 0 (8 products)
  3176  amazon kindle paper white ebook reader 4 gb 6 monochrome paper white t
   580  kindle voyage e reader 6 high resolution display 300 ppi with adaptive
   212  all new kind lee reader black 6 glare free touchscreen display wi fi i
    67  kindle oasis e reader with leather charging cover merlot 6 high resolu
    51  amazon kindle voyage 4 gb wi fi 3 g black
    30  kindle paper white e reader white 6 high resolution display 300 ppi wi
    15  kindle paper white
     6  kindle oasis e reader with leather charging cover black 6 high resolut

--- cluster 1 (13 products)
 10965  fire tablet 7 display wi fi 8 gb includes special offers magenta
  2814  all new fire hd 8 tablet 8 hd display wi fi 16 gb includes special off
  1685  fire kids edition tablet 7 display wi fi 16 gb green kid proof case
  1038  brand new amazon kindle fire 16 gb 7 ips display table twi fi 16 gb bl
   372  fire tablet 7 display wi fi 8 gb includes special offers black
   270  amazon

In [54]:
cluster_names = {
    0: "Kindle",
    1: "Tablets",
    2: "Echo & Smart Speakers",
    3: "Fire TV",
    4: "Chargers, Cables",
}
products["category"] = products["cluster"].map(cluster_names)

# Reading the printout in 3.3 shows one mistake: "Kindle Fire 16 GB Blue" is a Fire tablet, not an e-reader.
# Moving a wrong product by hand is a normal step after you read the clusters.
products.loc[products["product_name"].str.contains("Kindle Fire 16 GB"), "category"] = "Fire Tablets"

products[["product_name", "category"]].sort_values("category").sample(36)

,product_name,category
28545,amazon 9 w power fast official oem usb charger...,"Chargers, Cables"
3044,fire hd 8 tablet with alexa 8 hd display 32 gb...,Tablets
3459,all new kind lee reader black 6 glare free tou...,Kindle
34612,new amazon kindle fire hd 9 w power fast adapt...,"Chargers, Cables"
28618,amazon fire hd 6 standing protective case 4 th...,"Chargers, Cables"
28646,fire hd 10 tablet 101 hd display wi fi 16 gb i...,Tablets
29538,fire tv streaming media player 2015 model,Fire TV
21212,kindle paper white e reader white 6 high resol...,Kindle
21536,amazon fire 16 gb 5 th gen 2015 release black,Tablets
2886,all new fire hd 8 tablet 8 hd display wi fi 32...,Tablets
